# D1 — Power Diagnostics (run before believing any negative result)

Implements the project's rule of thumb as executable checks, in triage
order:

| # | Check | Catches | Cost |
|---|---|---|---|
| 1 | **Rows per input dimension** | starved fits (systematically pessimistic R²) | arithmetic |
| 2 | **Alignment (shuffle test)** | silent row misalignment — the classic pipeline bug | seconds |
| 3 | **Artifact sanity** | NaN/Inf, zero vectors, duplicates, broken normalization | seconds |

Runs on whatever artifacts exist in `DATA_DIR` (`activations.npz` from
Experiment A and/or `pairs.npz` from Experiment B) and prints PASS / FAIL
per check with the measured numbers.

**The logic of the shuffle test:** fit the same ridge map twice — once on
the data as-is, once with the target rows deliberately shuffled. If the
rows are truly aligned, the honest fit must beat the shuffled one
decisively (shuffled ≈ 0 or negative). If the two scores are close, the
"aligned" data was never aligned, and every downstream number is
meaningless. This doubles as a starvation detector: a strongly **negative**
shuffled score is the memorized-then-shrunk signature (run 1's −0.773).

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
import numpy as np
from pathlib import Path
DATA_DIR = Path(os.environ['DATA_DIR'])
rng = np.random.default_rng(0)
RESULTS = []

def verdict(name, ok, detail):
    RESULTS.append((name, ok))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {detail}")

def ridge(X, Y, a=1e-2):
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + a * np.eye(d), X.T @ Y)

def heldout_r2(X, Y, frac=0.75, a=1e-2):
    n = len(X)
    idx = rng.permutation(n)
    k = int(n * frac)
    tr, te = idx[:k], idx[k:]
    W = ridge(X[tr], Y[tr], a)
    P = X[te] @ W
    ss = ((Y[te] - P) ** 2).sum()
    st = ((Y[te] - Y[te].mean(0)) ** 2).sum()
    return 1 - ss / st

def check_power(X, Y, label):
    """#1: rows per input dimension (governing ratio)."""
    rows = int(len(X) * 0.75)
    spd = rows / X.shape[1]
    verdict(f"{label} · rows/input-dim", spd >= 5,
            f"{rows}/{X.shape[1]} = {spd:.1f} "
            f"(>=5 required, >=10 comfortable)")
    return spd

def check_alignment(X, Y, label):
    """#2: aligned fit must decisively beat a shuffled fit."""
    r_true = heldout_r2(X, Y)
    Ys = Y[rng.permutation(len(Y))]
    r_shuf = heldout_r2(X, Ys)
    ok = (r_true - r_shuf) > 0.2 and r_true > 0
    verdict(f"{label} · alignment (shuffle test)", ok,
            f"aligned R2={r_true:.3f} vs shuffled R2={r_shuf:.3f} "
            f"(gap {r_true - r_shuf:.3f}; need >0.2)")
    if r_shuf < -0.3:
        print(f"      note: strongly negative shuffled R2 "
              f"({r_shuf:.3f}) = memorize-then-shrink signature; "
              f"if the ALIGNED score is also low, suspect starvation "
              f"before absence of structure")

def check_artifact(M, label, expect_unit=False):
    """#3: NaN/Inf, zero rows, duplicates, normalization."""
    flat = M.reshape(-1, M.shape[-1])
    bad = int(np.isnan(flat).any(1).sum() + np.isinf(flat).any(1).sum())
    verdict(f"{label} · finite values", bad == 0,
            f"{bad} rows with NaN/Inf")
    norms = np.linalg.norm(flat, axis=1)
    nz = int((norms < 1e-6).sum())
    verdict(f"{label} · non-zero vectors", nz == 0, f"{nz} zero rows")
    sample = flat[rng.choice(len(flat), min(4000, len(flat)),
                             replace=False)]
    _, counts = np.unique(np.round(sample, 5), axis=0,
                          return_counts=True)
    dups = int((counts > 1).sum())
    verdict(f"{label} · duplicate rows", dups <= len(sample) * 0.01,
            f"{dups} duplicated rows in a {len(sample)}-row sample")
    if expect_unit:
        dev = float(np.abs(norms - 1).max())
        verdict(f"{label} · L2-normalized", dev < 1e-3,
                f"max |norm-1| = {dev:.2e}")

In [ ]:
# ---- Experiment A artifacts ----
f = DATA_DIR / 'activations.npz'
if f.exists():
    d = np.load(f)
    A, B = d['A_layers'], d['B_layers']
    print(f"activations.npz: A {A.shape}, B {B.shape}\n")
    L = A.shape[0] // 2                      # a middle layer
    X, Y = A[L].astype(np.float64), B[L].astype(np.float64)
    check_power(X, Y, f"ExpA L{L}")
    check_alignment(X, Y, f"ExpA L{L}")
    check_artifact(A, "ExpA A_layers")
    check_artifact(B, "ExpA B_layers")
    if 'R_layers' in d:
        check_artifact(d['R_layers'], "ExpA R_layers (control)")
else:
    print("activations.npz not found - skipping Experiment A checks")

In [ ]:
# ---- Experiment B artifacts ----
f = DATA_DIR / 'pairs.npz'
if f.exists():
    d = np.load(f)
    mob, sig = d['mob_img'].astype(np.float64), d['sig_img'].astype(np.float64)
    print(f"pairs.npz: mob {mob.shape}, sig {sig.shape}\n")
    check_power(mob, sig, "ExpB mob->sig")
    check_alignment(mob, sig, "ExpB mob->sig")
    check_artifact(mob, "ExpB mob_img", expect_unit=True)
    check_artifact(sig, "ExpB sig_img", expect_unit=True)
    if 'sig_txt' in d:
        check_artifact(d['sig_txt'], "ExpB sig_txt", expect_unit=True)
else:
    print("pairs.npz not found - skipping Experiment B checks")

print("\n" + "=" * 56)
fails = [n for n, ok in RESULTS if not ok]
if not RESULTS:
    print("NO ARTIFACTS FOUND - nothing checked")
elif not fails:
    print(f"ALL {len(RESULTS)} CHECKS PASS - negative results from these")
    print("artifacts can be interpreted as real absences of structure")
else:
    print(f"{len(fails)} CHECK(S) FAILED - do NOT interpret any negative")
    print("result until these are resolved:")
    for n in fails:
        print("  -", n)

## When to run this

1. **After every extraction** (a fresh A1 or B1 run) — before A2/A3/B2
   consume the artifacts. Catches alignment and sanity problems at the
   source instead of as confusing results two stages later.
2. **Before believing any negative or below-threshold result** — this is
   the primary purpose. Run-1's failure would have been caught here in
   seconds: rows/dim = 1.6 fails check #1, and the control's −0.773
   pattern is flagged by the shuffle test's note.
3. **After any pipeline change** — new dataset, new preprocessing, a
   model swap (e.g. MobileCLIP2 refit), fp16 storage, different pooling.
   Anything that touches how the matrices are produced.
4. **Not needed** after runs whose results you accept and whose pipeline
   is unchanged — the checks certify the measurement, not the science.

What it deliberately does not cover: check #3 of the rule of thumb
(pooling/preprocessing *choices*) can only be partially automated — this
notebook catches broken outputs (NaN, zero rows, un-normalized vectors),
but whether the preprocessing is the *right* one (identity normalization
on iOS, correct layer choice) remains a design review, not a script.